# Anki 批量制卡（Google Colab）

在 Colab 里可正常访问 OpenAI API。你需要：

1. 完整项目 ZIP（例如 GitHub 的 Download ZIP），包含 `anki_batch_generator.py` 和整个 `anki_generator/` 包（含 `resources/`）。不能只上传入口脚本。请勿打包密钥或私人数据。
2. `terms.json`：字符串数组，例如 `["word1", "word2"]`。
3. OpenAI API Key：推荐用 Colab **密钥**（左侧 🔑），名称为 `OPENAI_API_KEY`。

跑完后会生成 `anki_batch_output.apkg`，用最后一个单元格下载到电脑，再用 Anki 导入。

In [ ]:
# 依赖（Colab 已预装大部分，这里补齐制卡所需）
%pip install -q openai genanki requests

In [ ]:
# ========== 配置（按你的需要改）==========
MODE = "en_word"  # en_word | ja_word | interview | paper | interest
DECK_NAME = "English::Colab"
# 若账号暂无 gpt-5.x，可改为 "gpt-4o" 或 "gpt-4o-mini"
MODEL = "gpt-4o-mini"
TTS_MODEL = "gpt-4o-mini-tts"

In [ ]:
import os
from pathlib import Path

WORKDIR = Path("/content/anki_batch")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print("工作目录:", WORKDIR)

In [ ]:
from google.colab import files
from io import BytesIO
import tempfile
import zipfile

print("请上传可信的完整项目 ZIP，不要只上传入口脚本")
up = files.upload()
archives = [data for name, data in up.items() if name.lower().endswith(".zip")]
if len(archives) != 1:
    raise ValueError("请一次上传一个项目 ZIP")
project_dir = Path(tempfile.mkdtemp(prefix="project-", dir=WORKDIR)).resolve()
with zipfile.ZipFile(BytesIO(archives[0])) as archive:
    for member in archive.infolist():
        if not (project_dir / member.filename).resolve().is_relative_to(project_dir):
            raise ValueError("ZIP 包含不安全的文件路径")
    archive.extractall(project_dir)
entry_points = [
    path for path in project_dir.rglob("anki_batch_generator.py")
    if (path.parent / "anki_generator" / "__init__.py").is_file()
    and (path.parent / "anki_generator" / "application.py").is_file()
    and (path.parent / "anki_generator" / "resources" / "audio_bre_initial.svg").is_file()
]
if len(entry_points) != 1:
    raise ValueError("ZIP 必须包含一个完整项目（入口、anki_generator 包和资源）")
WORKDIR = entry_points[0].parent
os.chdir(WORKDIR)
print("项目目录:", WORKDIR)

In [ ]:
import json

# 方式 A：直接在这里写词条（JSON 数组）
TERMS = ["apologise", "burgeon"]

terms_path = WORKDIR / "terms.json"
terms_path.write_text(json.dumps(TERMS, ensure_ascii=False, indent=2), encoding="utf-8")
print("已写入", terms_path)

# 方式 B：若要用自己的文件，可注释掉上面，改为上传：
# up2 = files.upload()
# for name, data in up2.items():
#     (WORKDIR / "terms.json").write_bytes(data)

In [ ]:
import os

# 优先：Colab「密钥」里添加 OPENAI_API_KEY
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("已从 Colab userdata 读取 OPENAI_API_KEY")
except Exception as e:
    print("userdata 不可用:", e)
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("请输入 OpenAI API Key: ")

assert os.environ.get("OPENAI_API_KEY", "").strip(), "未设置 API Key"

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(WORKDIR / "anki_batch_generator.py"),
    "--mode", MODE,
    "--deck-name", DECK_NAME,
    "--terms-file", str(WORKDIR / "terms.json"),
    "--model", MODEL,
    "--tts-model", TTS_MODEL,
    "--output", str(WORKDIR / "anki_batch_output.apkg"),
    "--preview-json", str(WORKDIR / "anki_batch_preview.json"),
    "--cache-path", str(WORKDIR / "anki_batch_cache.json"),
    "--media-dir", str(WORKDIR / "anki_media"),
]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=str(WORKDIR))
if r.returncode != 0:
    raise RuntimeError(f"脚本退出码 {r.returncode}，请向上查看报错")


In [ ]:
from google.colab import files

apkg = WORKDIR / "anki_batch_output.apkg"
if apkg.is_file():
    files.download(str(apkg))
    print("已开始下载 anki_batch_output.apkg，请在 Anki 桌面端：文件 → 导入")
else:
    print("未找到 apkg，请检查上一格是否报错")